# H2O Implementation -- official H2O engine (submodule) + KVQuant-family harness -- 20% budget

This notebook evaluates H2O using the **official FMInference/H2O code from the
`H2O` submodule of the KVCacheCompression repo** -- specifically
`h2o_hf/utils_real_drop/modify_llama.py`'s `H2OKVCache_LayerWise`, the
authors' real-KV-dropping eviction engine (per-layer AND per-head heavy-hitter
scores, physical pruning of the cache tensors) -- inside the exact same
experimental harness as the KVQuant-family notebooks
(`KVQuant_2bit/3bit/4bit_Implementation.ipynb`,
`KVQuant_Baseline_Implementation.ipynb`): same model, same pinned package
versions, same seed, same dataset loading/tokenization/chunking, same GSM8K
prompt + question set + answer grading, same metric definitions, same
result-CSV schema.

This is one of three H2O eviction-budget variants (20% / 40% / 60% total
cache budget) -- identical in every other respect to
`H2O_official_submodule_Implementation.ipynb` (the 40% budget notebook) and
`H2O_official_submodule_60pct_Implementation.ipynb`. Only `HEAVY_RATIO` /
`RECENT_RATIO` and `METHOD_NAME` differ between the three.

**What comes from the submodule vs. what is harness code:**

- FROM THE SUBMODULE (unmodified): `H2OKVCache_LayerWise` -- heavy-hitter
  score accumulation (`_update_hh_score`: attention mass summed over batch
  and query rows, accumulated per head per cached position), the keep-set
  selection (per-head top-`hh_size` heavy hitters + last `recent_size`
  recent tokens), and the physical KV dropping. One engine instance per
  layer, exactly as `H2OLlamaAttention` instantiates it in the official
  code.
- HARNESS CODE (this notebook): the loop that runs the stock Llama-3.1-8B with
  `output_attentions=True` and feeds each layer's attention weights + legacy
  KV tuple through that layer's engine after every forward. The submodule's
  full `H2OLlamaAttention` module replacement could not be used directly
  because it targets an older transformers API (no `cache_position`, legacy
  tuple plumbing) and **assumes MHA** -- its cache-vs-score shapes break on
  GQA models like Llama-3.1-8B (8 KV heads vs. 32 query heads). Under the
  pinned `transformers==4.43.4` with eager attention, the stock model
  computes the identical attention math that class reimplements, so driving
  the official engine from the stock model's attention outputs preserves the
  method exactly.
- ONE GQA ADAPTATION (in harness code, submodule untouched): the engine
  expects one score row per cached KV head. Llama-3.1-8B's 32 query heads share
  8 KV heads (groups of 4), so each layer's attention weights are reduced to
  KV heads by summing the 4 query heads in each group -- the total attention
  mass received by that KV head's cache entries, the natural GQA
  generalization of the paper's per-head scores.

**Method requirements that legitimately differ from the KVQuant notebooks:**

1. `attn_implementation="eager"` -- H2O needs real attention weights;
   FlashAttention/SDPA cannot return them. Latency gaps vs. the sdpa
   notebooks therefore mix "eager tax" with "H2O tax"; run the baseline
   notebook once with eager to decompose them.
2. Memory here is **measured** (the engine physically drops KV positions);
   the quantized KVQuant notebooks report **calculated** bytes (simulated
   compression). Keep that distinction in mind when comparing memory columns.

**Budgets:** the official engine takes absolute `hh_size`/`recent_size`
counts (as in the repo's `run_summarization.py`). They are derived per sample
from ratios: `hh_size = int(0.10 * total_len)`, `recent_size = int(0.10 *
total_len)` -- i.e. a **20% total budget** (half of the other two H2O
notebooks' 40% and 60%), evenly split between the heavy-hitter and recency
components, matching that same even-split convention. Fresh engines (fresh
scores) are created per sample, mirroring the official per-request
`_clean_scores` usage.

**Harness parity (identical to the KVQuant family):** WikiText-103 is
constructed from the same complete test token stream and non-overlapping
2,048-token chunks, then up to **256 random chunks** are chosen with seed
**42**. GSM8K, ARC-Challenge, and HellaSwag are fully loaded and preprocessed
first, then up to **1,024 random valid examples per dataset** are chosen with
the same seed. Every method therefore receives the same reproducible subset.
Selected indices are sorted into source order before evaluation, avoiding an
arbitrary shuffled execution order while retaining a genuinely random subset.
All prompting, scoring, latency, perplexity, accuracy, memory aggregation,
and CSV-output formulas remain unchanged. On GSM8K, the reported H2O peak
still includes the transient dense prompt cache during batched prefill; the
steady-state post-prune peak is also printed.

Run cells top to bottom. Needs a GPU runtime. Before comparing across
notebooks, confirm the printed GPU name matches the other runs.

## Setup

In [ ]:
!hostname

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods -- kept
# byte-for-byte identical to the KVQuant-family notebooks (including
# datasets==2.14.5, which the previous H2O notebook left unpinned).

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) for consistency with the KVQuant-family notebooks,
# where the Fisher calibration cell needs them available early in Setup.
# This notebook has no such dependency itself, but keeping the split
# identical across the whole notebook family avoids Helper Functions
# containing different things in different notebooks. sync_if_cuda/
# clear_memory have no early dependency anywhere and live in Helper
# Functions with the rest of the genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Shared sampling policy used by every H2O and KVQuant comparison notebook:
#   - WikiText-103: up to 256 random non-overlapping chunks
#   - GSM8K, ARC-Challenge, HellaSwag: up to 1,024 random valid examples
#   - deterministic seed: 42
# Sampling happens after chunk construction or validity filtering. Selected
# indices are sorted back into source order so the subset is random while the
# evaluation order remains stable across methods.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
C = 2048
WIKITEXT_EVAL_SAMPLES = 256
QA_EVAL_SAMPLES = 1024

# Backward-compatible aliases used by a few downstream cells/comments.
N_EVAL_SAMPLES = WIKITEXT_EVAL_SAMPLES
N_QA_SAMPLES = QA_EVAL_SAMPLES

GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "h2o_budget_20pct"

# ---- H2O method hyperparameters (the only method-specific settings) ----
# The official engine receives absolute heavy-hitter and recent-token counts,
# derived per sample from the ratios below.
HEAVY_RATIO = 0.10
RECENT_RATIO = 0.10


random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)


def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)
    sample_count = min(int(max_samples), len(items))
    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )
    return [items[index] for index in selected_indices], selected_indices

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("Random sampling seed:", SHARED_SEED)
print("WikiText-103 random chunk target:", WIKITEXT_EVAL_SAMPLES)
print("GSM8K/ARC/HellaSwag random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("H2O heavy ratio:", HEAVY_RATIO, "| recent ratio:", RECENT_RATIO,
      "| total budget ratio:", HEAVY_RATIO + RECENT_RATIO)


In [ ]:
# Block - Load tokenizer + the stock (unmodified) model.
# Identical loading code and kwargs to the KVQuant-family notebooks, with ONE
# method-required difference: attn_implementation="eager". H2O needs the real
# per-step attention weights (output_attentions=True) to score heavy hitters,
# and neither FlashAttention nor SDPA can return them -- eager is a hard
# requirement of the method, not a harness choice. Everything else
# (torch_dtype, low_cpu_mem_usage, device_map, use_fast=False tokenizer,
# pad=eos) matches the other notebooks exactly.
#
# NOTE for cross-method latency analysis: because the KVQuant notebooks run
# sdpa, their timing gap vs. this notebook mixes "eager tax" with "H2O tax".
# Run the baseline notebook once with attn_implementation="eager" to separate
# the two.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "eager",   # H2O method requirement (see note above)
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading stock model (H2O modifies the cache at runtime, not the model)...")
model_h2o = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_h2o = model_h2o.to(DEVICE)
model_h2o.eval()
model_h2o.config.use_cache = True

device = next(model_h2o.parameters()).device
print("model_h2o loaded on:", device,
      "| dtype:", next(model_h2o.parameters()).dtype,
      "| attn: eager (required by H2O)")

In [ ]:
# Repo setup - Clone the KVCacheCompression repo (always fresh, matching the
# KVQuant notebooks' clean-clone convention) and initialize ONLY the H2O
# submodule -- the official FMInference/H2O repository. The other submodules
# (KVQuant, KIVI, SnapKV, ...) are not needed here, so a targeted init keeps
# this much faster than --recurse-submodules.
#
# From the submodule we import H2OKVCache_LayerWise from
# h2o_hf/utils_real_drop/modify_llama.py: the authors' real-KV-dropping
# eviction engine (per-layer, per-head heavy-hitter scores; physical pruning
# of the cache tensors). The module's other contents (H2OLlamaAttention /
# H2OLlamaForCausalLM) target an older transformers API and MHA-only models,
# so they are not used -- see the notebook intro for details.

if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone https://github.com/yoshikodes/KVCacheCompression.git /content/KVCacheCompression
%cd /content/KVCacheCompression
!git submodule update --init --depth 1 H2O

_h2o_module_path = "/content/KVCacheCompression/H2O/h2o_hf/utils_real_drop/modify_llama.py"
assert os.path.exists(_h2o_module_path), \
    "ERROR: official H2O module not found -- clone or submodule init may have failed."

sys.path.insert(0, "/content/KVCacheCompression/H2O/h2o_hf")
from utils_real_drop.modify_llama import H2OKVCache_LayerWise

print("Imported official H2O engine:", H2OKVCache_LayerWise,
      "\nfrom", _h2o_module_path)

## Helper Functions

Shared inference machinery used across all three datasets (WikiText-103,
GSM8K, ARC-Challenge).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - H2O machinery around the OFFICIAL engine -- shared inference
# machinery used across all three datasets (WikiText-103, GSM8K,
# ARC-Challenge). The eviction algorithm itself lives entirely in the
# submodule's H2OKVCache_LayerWise (imported above, unmodified): per-head
# heavy-hitter score accumulation, per-head top-k + recent keep-set
# selection, physical KV dropping. This cell only provides the plumbing
# that feeds it:
#
#   make_h2o_engines(total_len)
#       One fresh engine per layer (fresh hh_score), exactly as the official
#       H2OLlamaAttention holds one engine per layer. hh_size/recent_size are
#       derived from HEAVY_RATIO/RECENT_RATIO of this sample's total length.
#
#   reduce_attn_to_kv_heads(att)
#       The one GQA adaptation (submodule code untouched): the engine expects
#       one score row per cached KV head, so the 32 query heads' attention
#       weights are summed within each group of 8 sharing a KV head.
#
#   apply_engines(pkv, attentions, engines)
#       Feeds each layer's (legacy KV tuple, reduced attention) through that
#       layer's engine -- the same call H2OLlamaAttention.forward makes
#       (self.kv_cache(past_key_value, attn_weights)) -- and rebuilds the
#       legacy cache.
#
#   h2o_prefill(input_ids, engines)
#       ONE batched forward over the whole prompt (matching the batched
#       prefill inside model.generate() that the KVQuant notebooks use, and
#       matching how the official attention class natively handles
#       multi-token prefill: the full [.., q_len, kv_len] attention map goes
#       into the engine, whose _update_hh_score sums over the query rows),
#       then one engine pass (prune to budget).
#
#   h2o_step(token_tensor, abs_pos, pkv, engines)
#       One decode step: forward one token with explicit position_ids /
#       attention_mask (so RoPE positions stay absolute despite eviction --
#       cached keys were rotated at their original positions before caching,
#       so pruning preserves their correctness), then one engine pass.
#       Handles pkv=None (first step of the teacher-forced loop).
#
# resize_h2o_engines -- the one piece of GSM8K-specific machinery that
# would otherwise live here -- is defined in the GSM8K section instead,
# since it's only ever called from generate_gsm8k_h2o: GSM8K's real
# generation length isn't known in advance, unlike every other dataset
# here, which is teacher-forced over already-known tokens, so
# make_h2o_engines alone is enough for them.

import contextlib
import io

N_LAYERS = model_h2o.config.num_hidden_layers
N_KV_HEADS = int(getattr(model_h2o.config, "num_key_value_heads",
                         model_h2o.config.num_attention_heads))
N_Q_HEADS = model_h2o.config.num_attention_heads
KV_GROUP = N_Q_HEADS // N_KV_HEADS
print(f"Layers: {N_LAYERS} | query heads: {N_Q_HEADS} | KV heads: {N_KV_HEADS} "
      f"| GQA group size: {KV_GROUP}")


def cache_to_legacy(past_key_values):
    if past_key_values is None:
        return None
    if isinstance(past_key_values, tuple):
        return past_key_values
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


def get_cache_tokens(past_key_values):
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    return int(past_key_values[0][0].shape[2])


def dense_kv_cache_bytes(past_key_values):
    """Real bytes of the tensors actually resident in past_key_values right
    now, at their current dtype. The official engine physically drops
    positions, so this is an honest MEASURED number (the quantized KVQuant
    notebooks, by contrast, report CALCULATED bytes because their
    compression is simulated)."""
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return int(total)


def kv_bytes_per_token(past_key_values):
    """Exact real bytes one cached token occupies (uniform across positions:
    fp16 K + fp16 V for every layer). Used to convert a tracked max cache
    length into peak bytes without touching the GPU inside timed regions."""
    n_tokens = get_cache_tokens(past_key_values)
    if n_tokens == 0:
        return 0.0
    return dense_kv_cache_bytes(past_key_values) / n_tokens


def make_h2o_engines(total_len):
    """One fresh official engine per layer, budgets from this sample's total
    length. The engine's constructor print ("H2OKVCache-LayerWise: hh, rec")
    is suppressed -- it would fire n_layers times per sample; the budgets are
    reported once by the callers instead."""
    hh_size = max(1, int(total_len * HEAVY_RATIO))
    recent_size = max(1, int(total_len * RECENT_RATIO))
    with contextlib.redirect_stdout(io.StringIO()):
        engines = [
            H2OKVCache_LayerWise(hh_size=hh_size, recent_size=recent_size,
                                 k_seq_dim=2, v_seq_dim=2)
            for _ in range(N_LAYERS)
        ]
    return engines


def reduce_attn_to_kv_heads(att):
    """[batch, n_q_heads, q_len, kv_len] -> [batch, n_kv_heads, q_len, kv_len]
    by summing the query heads within each GQA group: the total attention
    mass received by that KV head's cache entries. (In HF Llama, query head h
    reads KV head h // KV_GROUP, i.e. groups are contiguous blocks.) On MHA
    models KV_GROUP == 1 and this is the identity."""
    if KV_GROUP == 1:
        return att
    b, h, q, kv = att.shape
    return att.view(b, N_KV_HEADS, KV_GROUP, q, kv).sum(dim=2)


def apply_engines(past_key_values, attentions, engines):
    """One official-engine call per layer -- the same call the official
    H2OLlamaAttention.forward makes: self.kv_cache(past_key_value,
    attn_weights). Updates that layer's per-head hh_score and physically
    prunes that layer's KV if over budget."""
    past_key_values = cache_to_legacy(past_key_values)
    new_layers = []
    for layer_kv, layer_att, engine in zip(past_key_values, attentions, engines):
        att_kv = reduce_attn_to_kv_heads(layer_att.detach())
        pruned = engine(tuple(layer_kv), att_kv)
        new_layers.append(tuple(pruned))
    return tuple(new_layers)


@torch.no_grad()
def h2o_prefill(input_ids, engines):
    """Returns (last_logits, pruned_pkv, prefill_cache_tokens).
    prefill_cache_tokens is the cache length BEFORE the post-prefill prune --
    the transient dense peak that batched-prefill H2O genuinely incurs."""
    outputs = model_h2o(
        input_ids=input_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )
    last_logits = outputs.logits[:, -1, :]
    pkv = cache_to_legacy(outputs.past_key_values)
    prefill_cache_tokens = get_cache_tokens(pkv)
    pkv = apply_engines(pkv, outputs.attentions, engines)
    return last_logits, pkv, prefill_cache_tokens


@torch.no_grad()
def h2o_step(token_tensor, abs_pos, past_key_values, engines):
    """One decode step through the H2O-compressed cache."""
    cache_len_before = get_cache_tokens(past_key_values)

    attention_mask = torch.ones(
        (1, cache_len_before + 1),
        dtype=torch.long,
        device=device,
    )
    position_ids = torch.tensor([[abs_pos]], dtype=torch.long, device=device)

    outputs = model_h2o(
        input_ids=token_tensor,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        position_ids=position_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    past_key_values = apply_engines(
        outputs.past_key_values, outputs.attentions, engines,
    )
    return outputs.logits[:, -1, :], past_key_values


_demo = make_h2o_engines(C)
print("Official H2O engines ready. Example budgets at total_len =", C, ":",
      f"hh_size {_demo[0].hh_size}, recent_size {_demo[0].recent_size},",
      f"cache_size {_demo[0].cache_size} tokens",
      f"({(_demo[0].cache_size / C):.1%} of dense)")
del _demo

In [ ]:
# Block - Shared multiple-choice (MC) scoring machinery, used by BOTH
# ARC-Challenge and HellaSwag. Mirrors the KVQuant-family notebooks'
# score_mc_question_kvquant exactly in structure and metric definitions;
# only the per-step mechanics differ, the same way score_chunk_h2o already
# differs from score_chunk_kvquant.
#
# lm_eval_encode_pair -- identical to the KVQuant-family notebooks (same
# joint-tokenization + truncation logic, matching lm-evaluation-harness).
#
# score_mc_choice_h2o(prompt, choice)
#     Times the full step-by-step walk over one choice's tokens via
#     h2o_step (mirroring score_chunk_h2o exactly -- eviction happens
#     INSIDE the timed bracket, since it's the method's real per-token
#     cost). Each choice gets its OWN independently-sized engine budget
#     from its OWN token length (make_h2o_engines(total_len) for this
#     choice) -- NOT a budget shared across a question's choices --
#     matching how every other dataset in this notebook sizes its H2O
#     budget as a ratio of that specific processed sequence's own true
#     length. Memory is the MEASURED peak cache size, tracked
#     incrementally via cheap shape reads outside the timing fences (same
#     technique as score_chunk_h2o), not a separate untimed pass.
#
# score_mc_question_h2o(prompt, choices, gold_index)
#     Same aggregation as score_mc_question_kvquant: raw/normalized
#     accuracy, perplexity from the gold choice only, TTFT = mean across
#     choices, TBT = weighted mean across every choice's decode steps,
#     total latency = SUM across choices, peak memory = MAX across
#     choices.


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")
    if len(continuation_ids) > int(C):
        raise ValueError(f"Choice has {len(continuation_ids)} tokens, exceeding C={C}.")

    max_context_tokens = int(C) + 1 - len(continuation_ids)
    if max_context_tokens < 1:
        raise ValueError("No room remains for a context token.")
    context_ids = context_ids[-max_context_tokens:]

    return context_ids, continuation_ids


@torch.no_grad()
def score_mc_choice_h2o(prompt, choice):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    engines = make_h2o_engines(full_ids_1d.shape[0])

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None
    max_cache_tokens = 0

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits, past_key_values = h2o_step(
            step_input, pos, past_key_values, engines,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

        max_cache_tokens = max(max_cache_tokens, get_cache_tokens(past_key_values))

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = int(round(max_cache_tokens * kv_bytes_per_token(past_key_values)))

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_h2o(prompt, choices, gold_index):
    choice_results = [score_mc_choice_h2o(prompt, choice) for choice in choices]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }

## WikiText-103

In [ ]:
# Block - WikiText-103 random sampling.
# Load the FULL test text, concatenate it into one token stream, split it into
# non-overlapping chunks of length C, then randomly select up to 256 chunks
# with seed 42. The random subset is identical across methods. Selected chunk
# indices are sorted into source order before evaluation so timing is not
# affected by an arbitrary shuffled execution order.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [text for text in testdata["text"] if len(text.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    all_chunks = chunk_sequence(ids, C)
    selected_chunks, selected_indices = seeded_subset(
        all_chunks,
        WIKITEXT_EVAL_SAMPLES,
        SHARED_SEED,
    )

    print(
        f"WikiText-103 test set: {len(texts)} non-empty lines, "
        f"{ids.shape[0]} tokens, {len(all_chunks)} available non-overlapping chunks"
    )
    print(
        f"WikiText-103 selected: {len(selected_chunks)} random chunks "
        f"(requested {WIKITEXT_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("WikiText-103 selected chunk indices (first 20):", selected_indices[:20])
    return selected_chunks


wikitext103_chunks = load_wikitext103_chunks()


In [ ]:
# Block - Step-by-step-per-chunk eval function.
#
# Mirrors the KVQuant notebooks' score_chunk_kvquant exactly: the model is
# stepped one token at a time using its own KV cache, teacher-forced (real
# corpus tokens fed in, never the model's own prediction), each step timed
# between the same cuda.synchronize() fences, NLL via the same
# CrossEntropyLoss on the step logits, next-token accuracy via argmax match.
# TTFT = the first step, TBT = mean of the remaining steps, total latency =
# sum of steps -- identical definitions and identical loop structure.
#
# The one method difference: each step goes through h2o_step, so the official
# engines' scoring + eviction happen INSIDE the timed bracket. That's
# deliberate: eviction is H2O's real per-token cost, exactly as
# QuantLinearSim's quantize/dequantize cost sits inside the KVQuant
# notebooks' timed forward.
#
# Fresh official engines (fresh per-head hh_scores) are created per chunk,
# mirroring the official per-request _clean_scores usage; budgets come from
# this chunk's length via the shared ratios.
#
# Memory: the engines physically prune the cache, so the per-chunk value is
# the MEASURED peak -- max cache length seen across steps (a cheap shape read
# done OUTSIDE the timing fences, so it cannot affect any timing number)
# converted to real fp16 tensor bytes.


@torch.no_grad()
def score_chunk_h2o(chunk_ids_1d):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    engines = make_h2o_engines(L)

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None
    max_cache_tokens = 0

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits, past_key_values = h2o_step(
            step_input, pos, past_key_values, engines,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

        # Shape read only -- outside the timing fences, no GPU work.
        max_cache_tokens = max(max_cache_tokens, get_cache_tokens(past_key_values))

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = int(round(max_cache_tokens * kv_bytes_per_token(past_key_values)))

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "max_cache_tokens": max_cache_tokens,
    }


@torch.no_grad()
def preview_chunk_prediction(chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate pass over just the first n_preview_tokens of a chunk
    -- for printing what the model actually predicted vs. the real next
    tokens. Runs through the same h2o_step machinery (with fresh engines
    budgeted from the full chunk length, same as the scored run) so the
    preview reflects the method's behavior. Does not affect any reported
    metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    L = chunk_ids_1d.shape[0]
    engines = make_h2o_engines(L)
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    past_key_values = None
    preds = []
    for pos in range(input_ids.shape[1]):
        step_logits, past_key_values = h2o_step(
            input_ids[:, pos:pos + 1], pos, past_key_values, engines,
        )
        preds.append(int(step_logits.argmax(dim=-1)[0].item()))

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds, skip_special_tokens=True),
    }

In [ ]:
# Block - WikiText-103 driver: run every chunk through score_chunk_h2o.
# Aggregation is identical to the KVQuant notebooks: perplexity =
# exp(pooled mean NLL over all scored tokens), accuracy = pooled next-token
# accuracy, TTFT/TBT/latency = means over chunks, peak_memory_gb = max over
# chunks, average_memory_gb = mean over chunks. Same output columns.


def evaluate_lm_dataset_h2o(dataset_name, chunks, method_label):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []
    max_cache_tokens_seen = 0
    per_chunk_records = []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_h2o(chunk_ids)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        max_cache_tokens_seen = max(max_cache_tokens_seen, result["max_cache_tokens"])

        chunk_scored = result["scored"]
        per_chunk_records.append({
            "chunk_index": chunk_idx,
            "prompt": tokenizer.decode(chunk_ids, skip_special_tokens=True),
            "accuracy": (result["correct"] / chunk_scored) if chunk_scored > 0 else float("nan"),
            "perplexity": math.exp(min(result["nll_sum"] / chunk_scored, 50.0)) if chunk_scored > 0 else float("nan"),
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    print(f"\n{dataset_name}: max retained cache across chunks = {max_cache_tokens_seen} tokens "
          f"(budget ratio {HEAVY_RATIO + RECENT_RATIO}; memory is MEASURED from the physically pruned cache)")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_wikitext103_per_prompt.csv"
    pd.DataFrame(per_chunk_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_chunk_records)} per-chunk {dataset_name} rows to {_per_prompt_path}")

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_h2o(_name, _chunks, METHOD_NAME))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

## GSM8K

In [ ]:
# Block - GSM8K loading: canonical test split, then random sampling.
# Build the complete list of valid question/answer pairs first, then select up
# to 1,024 examples with seed 42. This guarantees every method sees the same
# reproducible random questions rather than the first examples in the split.


def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

all_gsm8k_pairs = []
for item in gsm8k_test:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)
print("GSM8K selected valid-item indices (first 20):", gsm8k_selected_indices[:20])


In [ ]:
# Block - GSM8K generation with H2O (official engines). Prefill processes the
# whole prompt in ONE batched forward (exactly like the prefill inside
# model.generate() that the KVQuant notebooks use, and exactly how the
# official H2OLlamaAttention handles multi-token inputs: the full
# [.., q_len, kv_len] attention map goes into the engine, whose
# _update_hh_score sums over the query rows); decode then continues greedily
# one token at a time with per-step eviction.
#
# Budget sizing: unlike WikiText-103/ARC-Challenge/HellaSwag (which know
# their true sequence length upfront and size a FIXED engine budget from
# it), GSM8K's real generation length isn't known until generation is
# already done. So the budget is sized from a RUNNING AVERAGE of every
# PREVIOUSLY completed question's own true total length (its
# total_tokens = prompt_len + n_generated, only knowable in hindsight) --
# passed in as sizing_basis by the driver, which tracks that running
# average across qa_pairs. The very first question has no history yet, so
# it falls back to its own prompt_len alone (sizing_basis=None). The
# budget is computed ONCE per question via make_h2o_engines(sizing_basis)
# and held FIXED for that question's entire prefill + decode -- there is
# no more per-step resizing against this question's own live running
# length, since that would defeat the point of sizing from history rather
# than reacting to what has already happened within the same question.
#
# Timing matches the KVQuant notebooks' definitions exactly:
#   TTFT  = time from generation start until the first generated token's
#           logits are ready (i.e., the batched prefill forward) -- the same
#           quantity their StoppingCriteria timestamp captures.
#   total = wall time of the whole generation.
#   TBT   = (total - TTFT) / max(n_generated - 1, 1) -- the same formula.
# The post-prefill engine pass lands AFTER the TTFT timestamp (the first
# token comes straight from the prefill logits and does not need the prune),
# so it is counted in total latency / TBT as decode-phase method overhead.
#
# n_generated counts every token an actual timed forward call produced
# (prefill's token + one per h2o_step call), INCLUDING a final EOS token
# that ends the loop -- matching the KVQuant notebooks' StoppingCriteria,
# which fires after the EOS-producing step completes too, before
# generate() checks for termination. generated_ids/gen_text deliberately
# excludes EOS itself (matching skip_special_tokens=True decoding
# elsewhere), but n_generated must still count that step's forward call,
# or tbt_sec's denominator undercounts by one relative to KVQuant every
# time a question ends via EOS rather than hitting GSM8K_MAX_NEW_TOKENS --
# tracked separately below as n_forward_tokens.
#
# Answer grading is VERBATIM from the KVQuant notebooks: truncate the
# generation at the first "Question:", then extract "#### <num>" (falling
# back to the last number), compare |pred - gold| < 1e-4.
#
# Memory: MEASURED peak = max cache length observed, tracked via cheap
# shape reads taken ONLY after apply_engines has pruned that step's cache
# -- never the transient, budget-independent moment that exists right
# after the batched prefill forward but before eviction has run (a single
# forward over the whole prompt necessarily computes a full, unpruned
# cache for every prompt token, since H2O needs the resulting attention
# scores to decide what to evict in the first place -- that computation is
# unavoidable, but nothing downstream ever stores or reuses that
# pre-eviction snapshot: pkv is replaced by the pruned version immediately
# after). This matches exactly how WikiText-103/ARC-Challenge/HellaSwag
# already measure peak (they never have a pre-eviction moment to begin
# with, since they evict from the very first token), so peak/average
# memory are directly comparable, and budget-sensitive, across all four
# datasets.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same hand-rolled decode loop -- no separate teacher-forced pass.
# Per-step logits (last_logits) are already computed for greedy decoding
# regardless, so detecting the answer span costs one extra cheap text
# decode per token (inline, can't be deferred in a hand-rolled loop); the
# actual log-probability scoring is deferred until after gen_end below,
# using logits saved during the loop -- no extra forward pass, and no
# additional cost inside the timed window beyond that cheap text check.
# The span starts right after four consecutive '#' characters (the
# "####" marker GSM8K answers use) and stops the moment another '#'
# appears. If "####" never appears, perplexity is None for that question.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


@torch.no_grad()
def generate_gsm8k_h2o(question, sizing_basis=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    engines = make_h2o_engines(int(round(sizing_basis)) if sizing_basis is not None else prompt_len)

    sync_if_cuda()
    gen_start = time.perf_counter()

    # ---- Prefill (one batched forward; first token's logits ready here).
    # Done inline rather than via h2o_prefill so the TTFT timestamp can land
    # between the forward and the engine pass -- see header comment. ----
    outputs = model_h2o(input_ids=enc["input_ids"], use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    # ---- Official-engine pass on the prefill attention map, then prune
    # (counted in total latency, not TTFT -- see header comment) ----
    pkv = cache_to_legacy(outputs.past_key_values)
    pkv = apply_engines(pkv, outputs.attentions, engines)

    max_cache_tokens = get_cache_tokens(pkv)  # already post-eviction -- see header comment
    generated_ids = []
    n_forward_tokens = 1  # the prefill forward already produced next_id -- see header comment

    # ---- Live perplexity-span detection state -- see header comment ----
    hash_streak = 0
    in_answer_span = False
    answer_done = False
    answer_token_ids = []
    answer_logits = []

    # ---- Greedy decode with per-step eviction ----
    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)

        if not answer_done:
            tok_text = tokenizer.decode([next_id])
            if in_answer_span:
                if "#" in tok_text:
                    answer_done = True
                else:
                    answer_token_ids.append(next_id)
                    answer_logits.append(last_logits.detach())
            else:
                for ch in tok_text:
                    hash_streak = hash_streak + 1 if ch == "#" else 0
                if hash_streak >= 4:
                    in_answer_span = True

        if step == GSM8K_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = h2o_step(token_tensor, abs_pos, pkv, engines)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

        cache_tokens = get_cache_tokens(pkv)  # shape read only, already post-eviction
        max_cache_tokens = max(max_cache_tokens, cache_tokens)

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    # Perplexity of the model's own predicted final number, scored here
    # (fully after gen_end, untimed) from the logits saved above.
    if answer_token_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(answer_token_ids, answer_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(answer_token_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    bpt = kv_bytes_per_token(pkv)
    peak_bytes = int(round(max_cache_tokens * bpt))

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
        "perplexity": perplexity,
    }

In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_h2o
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass). H2O's budget for
# each question is sized from the running average of every PREVIOUSLY
# completed question's own true total length (total_tokens) -- tracked
# here as sizing_len_sum/sizing_len_count and passed to
# generate_gsm8k_h2o as sizing_basis (None for the first question, which
# falls back to its own prompt_len -- see that function's header).
# Aggregation is identical to the KVQuant notebooks: accuracy = fraction
# correct, perplexity = MEAN of per-question perplexities, TTFT/TBT/latency
# = means, peak_memory_gb = max over questions, average_memory_gb = mean
# over questions. Same output columns.


def evaluate_gsm8k_h2o(qa_pairs, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    sizing_len_sum = 0
    sizing_len_count = 0
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        sizing_basis = (sizing_len_sum / sizing_len_count) if sizing_len_count > 0 else None
        result = generate_gsm8k_h2o(qa["question"], sizing_basis)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        sizing_len_sum += result["total_tokens"]
        sizing_len_count += 1

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "prompt": qa["question"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": ppl,
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_h2o(gsm8k_qa_pairs, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

## ARC-Challenge

Likelihood-based multiple-choice scoring (no generation): every answer
choice is scored (raw and character-length-normalized), perplexity from
the gold choice, TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - ARC-Challenge loading: canonical test split, validity filtering,
# then random sampling. Invalid answer-key rows are removed before selecting
# up to 1,024 valid questions with seed 42.


def load_arc_challenge_items():
    testdata = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )

    valid_items = []
    for row in testdata:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available; "
        f"selected {len(selected_items)} random questions "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_h2o (Helper Functions), then
# reports character-length normalized MC accuracy. Aggregation mirrors
# GSM8K: perplexity = MEAN of per-question perplexities (from each
# question's gold choice), TTFT/TBT/latency = means over questions,
# peak_memory_gb = max over questions, average_memory_gb = mean over
# questions.


def evaluate_arc_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_h2o(prompt, choice_texts, gold_index)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["question"],
            "choices": item["choices"],
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_h2o(arc_items, METHOD_NAME),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)

## HellaSwag

Likelihood-based multiple-choice scoring (no generation), same
methodology as ARC-Challenge: every answer choice is scored (raw and
character-length-normalized), perplexity from the gold ending,
TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - HellaSwag loading: canonical validation split, preprocessing,
# then random sampling. HellaSwag's test labels are private, so validation is
# the standard evaluation split. Up to 1,024 processed examples are selected
# with seed 42.


def _hellaswag_preprocess(text):
    """Match lm-evaluation-harness HellaSwag preprocessing."""
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def load_hellaswag_items():
    dataset = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )

    processed_items = []
    for row in dataset:
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = _hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [_hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} validation examples available; "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_h2o (Helper Functions) -- the same
# machinery ARC-Challenge uses, since both are likelihood-based
# multiple-choice datasets. Aggregation is identical to ARC-Challenge:
# normalized accuracy, perplexity = MEAN of per-question perplexities
# (from each example's gold ending), TTFT/TBT/latency = means over
# examples, peak_memory_gb = max over examples, average_memory_gb =
# mean over examples.


def evaluate_hellaswag_h2o(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_h2o(item["prompt"], item["choices"], item["gold_index"])

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["prompt"],
            "choices": item["choices"],
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_h2o(hellaswag_items, METHOD_NAME),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)

## Save Results

In [ ]:
# Block - Combine WikiText-103 / GSM8K / ARC-Challenge / HellaSwag results into
# one table with ONLY the specified metrics, then save to CSV -- identical
# column schema to the KVQuant-family notebooks, saved into the same Drive
# folder, so all methods' CSVs concatenate directly into one comparison
# table.
#
# Cross-method comparison reminders:
#   - This notebook's memory numbers are MEASURED (physically pruned cache);
#     the quantized KVQuant notebooks' are CALCULATED (simulated
#     compression). Say so when presenting them side by side.
#   - This notebook runs eager attention (H2O method requirement); the
#     KVQuant notebooks run sdpa. To decompose "eager tax" vs "H2O tax" in
#     the latency columns, also run the baseline notebook once with
#     attn_implementation="eager".
#   - Confirm the GPU name printed in Block 2 matches every other notebook's
#     run before trusting any timing/memory comparison.
#
# Robust to partial runs: only concatenates whichever of
# lm_results_df / gsm8k_results_df / arc_results_df / hellaswag_results_df
# actually exist in this session, so you can run just a subset of
# datasets' cells without this cell crashing on a NameError for a
# dataframe you never created.

_result_df_names = ["lm_results_df", "gsm8k_results_df", "arc_results_df", "hellaswag_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{METHOD_NAME}_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")